# Airtable Setup

**Run this notebook once every time you start a fresh base.**

It creates (or resets) both tables and both journal form views in one pass.

## What it does

| Step | Action |
|---|---|
| 1 | Load credentials from `.env` |
| 2 | Define helpers: truncate, create_table, setup_table |
| 3 | Define TradeLog schema (20 fields, `ticker` as primary) |
| 4 | Define Journal schema (24 fields, `date` as primary) |
| 5 | Create or reset both tables |
| 6 | Journal form instructions (one-time manual setup in Airtable UI) |
| 7 | Print table IDs and next steps |

## Notes

- **Safe to re-run.** If a table already exists its records are deleted;
  the schema is left intact (Airtable Meta API has no table-delete endpoint).
- **Form views** cannot be created via the API â€” create them once in the
  Airtable UI (Cell 6 has the exact field lists). They survive table resets.
- **autoNumber fields** cannot be created via the API either â€” add a TradeID
  or JournalID column manually in the UI after first run if you want one.

## 1 Â· Credentials

Reads `AIRTABLE_API_KEY` and `AIRTABLE_BASE_ID` from `.env`.
Only those two keys are required here; `AIRTABLE_TABLE` is optional
(it references the TradeLog table ID, printed at the end of this notebook).

In [ ]:
import os, requests
from pathlib import Path

def load_env(path='.env'):
    if not Path(path).exists():
        raise FileNotFoundError('No .env -- copy .env.example and fill in credentials.')
    with Path(path).open() as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line: continue
            k, _, v = line.partition('=')
            os.environ.setdefault(k.strip(), v.strip())

load_env()
missing = [k for k in ('AIRTABLE_API_KEY', 'AIRTABLE_BASE_ID') if not os.getenv(k)]
if missing: raise EnvironmentError(f'Missing in .env: {missing}')

BASE_ID    = os.getenv('AIRTABLE_BASE_ID')
META_URL   = f'https://api.airtable.com/v0/meta/bases/{BASE_ID}'
DATA_URL   = f'https://api.airtable.com/v0/{BASE_ID}'
AT_HEADERS = {
    'Authorization': f"Bearer {os.getenv('AIRTABLE_API_KEY')}",
    'Content-Type':  'application/json',
}
print('credentials OK')
print(f'base: {BASE_ID}')

## 2 Â· Helpers

Four utility functions that wrap the Airtable Meta and Data APIs:

**`existing_tables()`** â€” calls the Meta API to list every table in the base.
Returns a `{name: table_object}` dict. The table object contains the field
list with IDs, which we need later for form view creation.

**`truncate(table_name)`** â€” deletes all records from a table using the
Data API's batch-delete endpoint (10 at a time). This is the workaround for
the Meta API's missing DELETE-table endpoint.

**`create_table(name, fields)`** â€” POSTs a new table to the Meta API.
Returns the full table object including all field IDs.

**`setup_table(name, fields)`** â€” orchestrates the above: truncates if
the table exists (preserving schema), creates from scratch if it does not.
Returns the table object either way.

In [ ]:
def existing_tables():
    r = requests.get(f'{META_URL}/tables', headers=AT_HEADERS)
    r.raise_for_status()
    return {t['name']: t for t in r.json().get('tables', [])}


def truncate(table_name):
    url, deleted = f'{DATA_URL}/{table_name}', 0
    while True:
        r = requests.get(url, headers=AT_HEADERS, params={'maxRecords': 10})
        r.raise_for_status()
        recs = r.json().get('records', [])
        if not recs: break
        ids    = [rec['id'] for rec in recs]
        params = '&'.join(f'records[]={i}' for i in ids)
        requests.delete(f'{url}?{params}', headers=AT_HEADERS).raise_for_status()
        deleted += len(ids)
    print(f'  CLEARED {table_name!r}  ({deleted} records deleted)')


def create_table(name, fields):
    r = requests.post(f'{META_URL}/tables', headers=AT_HEADERS,
                      json={'name': name, 'fields': fields})
    r.raise_for_status()
    return r.json()


def setup_table(name, fields):
    tables = existing_tables()
    if name in tables:
        truncate(name)
        print(f'  EXISTS  {name!r}  -- schema kept, records cleared')
        return tables[name]
    tbl = create_table(name, fields)
    print(f'  CREATED {name!r}  id={tbl["id"]}')
    return tbl


def get_field_ids(table_obj):
    """Return {field_name: field_id} for all fields in a table object."""
    return {f['name']: f['id'] for f in table_obj.get('fields', [])}

## 3 Â· TradeLog schema

20 fields covering the full trade lifecycle from order to exit.
The primary field (`ticker`) is a plain text field â€” Airtable's Meta API
does not support `autoNumber` as primary; add a TradeID column manually
in the UI if you want a sequential ID.

| Group | Fields |
|---|---|
| Identity | ticker (primary), strategy, broker |
| Order | entry_date, order_type, quantity_order, price_order, stop_price, target_price, risk_pct |
| Execution | quantity_exec, price_exec, commission, record_id |
| Exit | exit_date, price_exit, gross_pnl, net_pnl, trade_result |
| Flags | missed_trade |

In [ ]:
TRADELOG_PRIMARY = {'name': 'ticker', 'type': 'singleLineText'}

TRADELOG_FIELDS = [
    # Order metadata
    {'name': 'entry_date',    'type': 'date',
     'options': {'dateFormat': {'name': 'iso'}}},
    {'name': 'strategy',      'type': 'singleSelect',
     'options': {'choices': [
         {'name': 'momentum'}, {'name': 'trend'},
         {'name': 'mean_reversion'}, {'name': 'event'},
     ]}},
    {'name': 'broker',        'type': 'singleLineText'},
    {'name': 'order_type',    'type': 'singleSelect',
     'options': {'choices': [
         {'name': 'limit'}, {'name': 'market'},
         {'name': 'stop'}, {'name': 'stop_limit'},
     ]}},
    {'name': 'quantity_order','type': 'number', 'options': {'precision': 0}},
    {'name': 'price_order',   'type': 'number', 'options': {'precision': 4}},
    {'name': 'stop_price',    'type': 'number', 'options': {'precision': 4}},
    {'name': 'target_price',  'type': 'number', 'options': {'precision': 4}},
    {'name': 'risk_pct',      'type': 'number', 'options': {'precision': 2}},
    # Execution
    {'name': 'quantity_exec', 'type': 'number', 'options': {'precision': 0}},
    {'name': 'price_exec',    'type': 'number', 'options': {'precision': 4}},
    {'name': 'commission',    'type': 'number', 'options': {'precision': 2}},
    {'name': 'record_id',     'type': 'singleLineText'},
    # Exit fields (patched by _close_fifo)
    {'name': 'exit_date',     'type': 'date',
     'options': {'dateFormat': {'name': 'iso'}}},
    {'name': 'price_exit',    'type': 'number', 'options': {'precision': 4}},
    {'name': 'gross_pnl',     'type': 'number', 'options': {'precision': 2}},
    {'name': 'net_pnl',       'type': 'number', 'options': {'precision': 2}},
    {'name': 'trade_result',  'type': 'singleSelect',
     'options': {'choices': [
         {'name': 'win'}, {'name': 'loss'}, {'name': 'breakeven'},
     ]}},
    # Classification
    {'name': 'missed_trade',  'type': 'singleSelect',
     'options': {'choices': [
         {'name': 'missed'}, {'name': 'override'},
     ]}},
]

print(f'TradeLog schema: 1 primary + {len(TRADELOG_FIELDS)} fields = {1+len(TRADELOG_FIELDS)} total')

## 4 Â· Journal schema

24 fields for the daily psychology journal. The primary field is `date`
(stored as text `YYYY-MM-DD` so form submissions always include it).

| Group | Fields | Source |
|---|---|---|
| Identity | date (primary), session | both forms |
| Morning | mindset_pre, confidence_pre, checklist_done, perf_forecast, bc_1, bc_2 | Morning form |
| Evening | mindset_post, confidence_post, perf_actual | Evening form |
| Review | what_went_well_1/2/3, tomorrows_kaizen, notes | Evening form |
| Gratitude | gratitude_1/2/3 | Evening form |
| Computed | bulls_eye, score, streak, multiplier, final_score | Part B notebook |

The five computed fields are left blank at entry time and patched by the
Part B notebook each time it runs.

In [ ]:
JOURNAL_PRIMARY = {'name': 'date', 'type': 'singleLineText'}

JOURNAL_FIELDS = [
    # Session identifier â€” determines which form row this is
    {'name': 'session',          'type': 'singleSelect',
     'options': {'choices': [{'name': 'pre'}, {'name': 'post'}]}},
    # â”€â”€ Morning Check-in fields â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    {'name': 'mindset_pre',      'type': 'number', 'options': {'precision': 0}},
    {'name': 'confidence_pre',   'type': 'number', 'options': {'precision': 0}},
    {'name': 'checklist_done',   'type': 'number', 'options': {'precision': 0}},
    {'name': 'perf_forecast',    'type': 'number', 'options': {'precision': 1}},
    {'name': 'bc_1',             'type': 'singleLineText'},  # bias / conviction 1
    {'name': 'bc_2',             'type': 'singleLineText'},  # bias / conviction 2
    # â”€â”€ Evening Review fields â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    {'name': 'mindset_post',     'type': 'number', 'options': {'precision': 0}},
    {'name': 'confidence_post',  'type': 'number', 'options': {'precision': 0}},
    {'name': 'perf_actual',      'type': 'number', 'options': {'precision': 1}},
    {'name': 'what_went_well_1', 'type': 'singleLineText'},
    {'name': 'what_went_well_2', 'type': 'singleLineText'},
    {'name': 'what_went_well_3', 'type': 'singleLineText'},
    {'name': 'tomorrows_kaizen', 'type': 'multilineText'},
    {'name': 'notes',            'type': 'multilineText'},
    # â”€â”€ Gratitude â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    {'name': 'gratitude_1',      'type': 'singleLineText'},
    {'name': 'gratitude_2',      'type': 'singleLineText'},
    {'name': 'gratitude_3',      'type': 'singleLineText'},
    # â”€â”€ Computed by Part B notebook (do not fill manually) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    {'name': 'bulls_eye',        'type': 'number', 'options': {'precision': 0}},
    {'name': 'score',            'type': 'number', 'options': {'precision': 0}},
    {'name': 'streak',           'type': 'number', 'options': {'precision': 0}},
    {'name': 'multiplier',       'type': 'number', 'options': {'precision': 2}},
    {'name': 'final_score',      'type': 'number', 'options': {'precision': 1}},
]

print(f'Journal schema: 1 primary + {len(JOURNAL_FIELDS)} fields = {1+len(JOURNAL_FIELDS)} total')

## 5 Â· Create or reset tables

Calls `setup_table` for both tables in sequence. Each call:

- **Table exists** â†’ clears all records, returns the existing table object
  (field schema is preserved â€” no need to re-enter your column order or widths)
- **Table missing** â†’ creates it from the schema defined above

The returned table objects are kept in `tradelog_obj` and `journal_obj`
because we need the field IDs to create the form views in the next cell.

In [ ]:
print('=== Setting up TradeLog ===')
tradelog_obj = setup_table('TradeLog', [TRADELOG_PRIMARY] + TRADELOG_FIELDS)

print()
print('=== Setting up journal ===')
journal_obj  = setup_table('journal',  [JOURNAL_PRIMARY]  + JOURNAL_FIELDS)

tradelog_id = tradelog_obj['id']
journal_id  = journal_obj['id']
journal_fids = get_field_ids(journal_obj)

print()
print(f'TradeLog id : {tradelog_id}')
print(f'journal  id : {journal_id}')
print(f'journal fields mapped: {len(journal_fids)}')

## 6 Â· Journal form views

The Airtable Meta API does not support creating form views programmatically â€”
the endpoint exists but always returns 422. Forms are created **once** in the
Airtable UI and survive table resets (only deleting the table itself kills them).

**Create them once after a fresh base setup:**

1. Open the `journal` table â†’ **+ Add a view â†’ Form**
2. Create **Morning Check-in** â€” show only:
   `date Â· session Â· mindset_pre Â· confidence_pre Â· checklist_done Â· perf_forecast Â· bc_1 Â· bc_2`
3. Create **Evening Review** â€” show only:
   `date Â· session Â· mindset_post Â· confidence_post Â· perf_actual Â·
   what_went_well_1/2/3 Â· tomorrows_kaizen Â· notes Â· gratitude_1/2/3`
4. For each form: **Share form** â†’ copy the link â†’ bookmark it on your phone

These two links are your only daily data-entry interface.
The notebook never needs to touch them again.

## 7 Â· Summary

Confirms both tables are ready and prints the TradeLog table ID.
Update `AIRTABLE_TABLE` in your `.env` if the ID changed.

In [ ]:
print('=' * 60)
print('SETUP COMPLETE')
print('=' * 60)

print(f'\nTradeLog id : {tradelog_id}')
print(f'journal  id : {journal_id}')

print()
print('Update .env if TradeLog id changed:')
print(f'  AIRTABLE_TABLE={tradelog_id}')

print()
print('Next: create two form views manually in Airtable (see Cell 6).')
print('Then bookmark the share links and start your daily journal.')

print()
print('Daily workflow:')
print('  Before open  ->  Morning Check-in form  (~2 min)')
print('  After close  ->  Evening Review form     (~5 min)')
print('  After close  ->  run trading-journal-v5 Part A')
print('  Weekly       ->  run trading-journal-v5 Part B')